# Customer Support Memory with Engram

Build a support agent that remembers past customer interactions. When a customer contacts support again, the agent instantly retrieves relevant history — no manual lookup needed.

**Use case**: A SaaS company's support team uses Engram to give agents full context on returning customers.

In [ ]:
# !pip install engram-search

In [ ]:
from engram.backends.faiss_backend import FaissBackend
from engram.backends.base import Document
from engram.retrieval.embedder import Embedder
from engram.retrieval.sparse import BM25
from engram.retrieval.pipeline import reciprocal_rank_fusion
from engram.ingestion.parser import session_to_documents

backend = FaissBackend(path="./support_store", dimension=1024)
embedder = Embedder("bge-large")

print("Support memory store initialized")

## 1. Ingest Support Ticket History

Imagine these are past support conversations for a customer named Sarah.

In [ ]:
support_tickets = [
    {
        "id": "ticket_1001",
        "timestamp": "2024-01-15",
        "customer": "sarah@acme.com",
        "turns": [
            {"role": "user", "content": "Hi, I'm having trouble connecting my Slack integration. It was working yesterday but now it shows 'authentication failed'."},
            {"role": "assistant", "content": "I can help with that. Let me check your integration settings. It looks like your OAuth token expired. I'll regenerate it for you."},
            {"role": "user", "content": "That fixed it, thanks! Also, can you make sure it posts to the #alerts channel instead of #general?"},
            {"role": "assistant", "content": "Done! I've updated the default channel to #alerts. You can change this anytime from Settings > Integrations > Slack."}
        ]
    },
    {
        "id": "ticket_1042",
        "timestamp": "2024-02-20",
        "customer": "sarah@acme.com",
        "turns": [
            {"role": "user", "content": "Our team just upgraded to the Enterprise plan. How do I set up SSO with Okta?"},
            {"role": "assistant", "content": "Congratulations on the upgrade! For Okta SSO, go to Admin > Security > SAML SSO. You'll need your Okta metadata URL."},
            {"role": "user", "content": "Got it. Also, we need to add 15 new team members. Is there a bulk invite option?"},
            {"role": "assistant", "content": "Yes! Go to Admin > Team > Bulk Import. You can upload a CSV with email addresses. Each user will get an invite email with SSO login instructions."}
        ]
    },
    {
        "id": "ticket_1089",
        "timestamp": "2024-03-10",
        "customer": "sarah@acme.com",
        "turns": [
            {"role": "user", "content": "We're seeing slow dashboard load times since last week. Sometimes it takes 10+ seconds."},
            {"role": "assistant", "content": "I see the issue. Your account has over 50,000 records and the default view is loading all of them. Let me enable pagination for your workspace."},
            {"role": "user", "content": "That helped a lot! Can we also set a default date filter so it only shows last 30 days?"},
            {"role": "assistant", "content": "Absolutely. I've set the default filter to 'Last 30 days' for all dashboard views. Your team members will see this change on their next login."}
        ]
    },
    {
        "id": "ticket_1150",
        "timestamp": "2024-03-28",
        "customer": "bob@startup.io",
        "turns": [
            {"role": "user", "content": "How do I export my data to CSV? I need to create a report for our investors."},
            {"role": "assistant", "content": "Go to Reports > Export. Select the date range and metrics you need, then click 'Download CSV'. For investor reports, the 'Executive Summary' template might be useful."},
            {"role": "user", "content": "Perfect. Can I schedule this to run automatically every month?"},
            {"role": "assistant", "content": "Yes! In Reports > Scheduled Reports, create a new schedule. Set it to monthly and add recipient emails. The CSV will be sent automatically."}
        ]
    }
]

# Ingest all tickets
all_docs = []
for ticket in support_tickets:
    parsed = session_to_documents(
        session=ticket["turns"],
        session_id=ticket["id"],
        timestamp=ticket["timestamp"],
        include_assistant=True,
    )
    all_docs.extend(parsed)

texts = [d["text"] for d in all_docs]
embeddings = embedder.encode_documents(texts)

documents = [
    Document(id=d["id"], text=d["text"], embedding=embeddings[i].tolist(), metadata=d["metadata"])
    for i, d in enumerate(all_docs)
]
backend.add(documents)

print(f"Ingested {len(documents)} documents from {len(support_tickets)} tickets")

## 2. Customer Returns with a New Issue

Sarah contacts support again. Before the agent responds, Engram retrieves her full history.

In [ ]:
def get_customer_context(query, top_k=5):
    """Retrieve relevant past interactions for context."""
    # Dense retrieval
    query_vec = embedder.encode_query(query)
    dense_results = backend.query(embedding=query_vec.tolist(), top_k=top_k * 2)
    dense_ranking = [(d.id, d.score) for d in dense_results]
    
    # BM25 for keyword matching
    bm25 = BM25()
    all_texts = [d.text for d in documents]
    all_ids = [d.id for d in documents]
    bm25_scores = bm25.score_query_against_docs(query, all_texts)
    sparse_ranking = sorted(zip(all_ids, bm25_scores), key=lambda x: x[1], reverse=True)
    
    # Fuse with RRF
    fused = reciprocal_rank_fusion(dense_ranking, sparse_ranking)
    
    id_to_doc = {d.id: d for d in documents}
    results = []
    for doc_id, score in fused[:top_k]:
        if doc_id in id_to_doc:
            doc = id_to_doc[doc_id]
            results.append((doc, score))
    return results


# Sarah's new message
new_query = "The Slack integration stopped working again. Same error as before."

print(f'New ticket from Sarah: "{new_query}"')
print(f'\n{"=" * 60}')
print("Retrieved context from past tickets:\n")

context = get_customer_context(new_query)
for doc, score in context:
    meta = doc.metadata or {}
    print(f"  [{meta.get('session_id', '?')}] ({meta.get('timestamp', '')}) Score: {score:.4f}")
    print(f"  {doc.text[:150]}...")
    print()

## 3. Build the Agent Prompt

Use the retrieved context to build a prompt for any LLM. The agent now knows Sarah's full history.

In [ ]:
def build_support_prompt(customer_query, context):
    """Build a context-aware prompt for the support agent."""
    context_text = ""
    for doc, score in context:
        meta = doc.metadata or {}
        context_text += f"\n[{meta.get('session_id', '?')} - {meta.get('timestamp', '')}]\n"
        context_text += doc.text[:300] + "\n"
    
    prompt = f"""You are a customer support agent. Use the customer's past interaction history to provide personalized, context-aware support.

PAST INTERACTIONS:
{context_text}

CURRENT MESSAGE:
{customer_query}

Respond helpfully, referencing past interactions when relevant. Be specific about what was done before."""
    return prompt

prompt = build_support_prompt(new_query, context)
print(prompt)

## 4. Try More Queries

The agent can answer questions about any past interaction.

In [ ]:
queries = [
    "What plan is Sarah on?",
    "How do I set up SSO?",
    "Dashboard is slow",
    "How to export reports?",
]

for q in queries:
    print(f'\nQuery: "{q}"')
    results = get_customer_context(q, top_k=2)
    for doc, score in results:
        meta = doc.metadata or {}
        print(f"  -> {meta.get('session_id', '?')} ({meta.get('timestamp', '')}) [{score:.4f}]")
    print()

## Cleanup

In [ ]:
import shutil
shutil.rmtree("./support_store", ignore_errors=True)
print("Support store cleaned up.")